In [1]:

import geopandas as gpd
import pandas as pd

from pathlib import Path

In [2]:
# Read the zipped shapefile or geo data inside the zip into a GeoDataFrame
df_movement = pd.read_csv(f"movement-speeds-quarterly-by-hod-berlin-2019-Q2.csv.zip")

df_movement.head()

,year,quarter,hour_of_day,segment_id,start_junction_id,end_junction_id,osm_way_id,osm_start_node_id,osm_end_node_id,speed_kph_mean,speed_kph_stddev,speed_kph_p50,speed_kph_p85
0,2019,2,0,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,41.258,9.239,41.330,46.680
1,2019,2,23,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,35.794,6.706,35.832,41.398
2,2019,2,22,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,32.766,13.401,35.171,43.469
3,2019,2,23,277640ca389fc7f0fc8a583386e6063df80485f0,ad26106a25d52c3409bd0165f05de3047a4e96b5,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,81398215,1236497752,34.507,4.639,35.207,38.160
4,2019,2,1,277640ca389fc7f0fc8a583386e6063df80485f0,ad26106a25d52c3409bd0165f05de3047a4e96b5,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,81398215,1236497752,41.512,6.049,41.128,47.681


In [3]:
#get osm network

In [4]:
###

import osmium

In [5]:
import osmium
import geopandas as gpd
from shapely.geometry import LineString

class WaySegmentExtractor(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.segments = []

    def way(self, w):
        if 'highway' not in w.tags:
            return  # skip non-road

        if not w.nodes or len(w.nodes) < 2:
            return  # skip invalid ways

        coords = [(n.location.lon, n.location.lat) for n in w.nodes if n.location.valid()]
        node_ids = [n.ref for n in w.nodes]

        if len(coords) != len(node_ids):
            return

        for i in range(len(node_ids) - 1):
            line = LineString([coords[i], coords[i + 1]])
            self.segments.append({
                "osm_way_id": w.id,
                "osm_start_node_id": node_ids[i],
                "osm_end_node_id": node_ids[i + 1],
                "geometry": line
            })

# Run the handler
handler = WaySegmentExtractor()
handler.apply_file("berlin-200101.osm.pbf", locations=True)

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(handler.segments, geometry="geometry", crs="EPSG:4326")
print(gdf.head())


   osm_way_id  osm_start_node_id  osm_end_node_id  \
0     4045150         1234120411       6375730236   
1     4045150         6375730236        262876417   
2     4045150          262876417        262877047   
3     4045150          262877047         21432146   
4     4045150           21432146       6376074936   

                                            geometry  
0    LINESTRING (13.60517 52.37342, 13.6055 52.3735)  
1    LINESTRING (13.6055 52.3735, 13.60616 52.37366)  
2   LINESTRING (13.60616 52.37366, 13.6074 52.37398)  
3   LINESTRING (13.6074 52.37398, 13.60756 52.37403)  
4  LINESTRING (13.60756 52.37403, 13.60875 52.37443)  


In [6]:
gdf

,osm_way_id,osm_start_node_id,osm_end_node_id,geometry
0,4045150,1234120411,6375730236,"LINESTRING (13.60517 52.37342, 13.6055 52.3735)"
1,4045150,6375730236,262876417,"LINESTRING (13.6055 52.3735, 13.60616 52.37366)"
2,4045150,262876417,262877047,"LINESTRING (13.60616 52.37366, 13.6074 52.37398)"
3,4045150,262877047,21432146,"LINESTRING (13.6074 52.37398, 13.60756 52.37403)"
4,4045150,21432146,6376074936,"LINESTRING (13.60756 52.37403, 13.60875 52.37443)"
...,...,...,...,...
672448,760058780,7100175961,7100175959,"LINESTRING (13.44498 52.54839, 13.44562 52.54913)"
672449,760058780,7100175959,7100175958,"LINESTRING (13.44562 52.54913, 13.44637 52.54994)"
672450,760058781,7100175965,7100175964,"LINESTRING (13.44471 52.5484, 13.44476 52.54847)"
672451,760058781,7100175964,7100175963,"LINESTRING (13.44476 52.54847, 13.44494 52.54865)"


In [15]:
segments=gdf.copy()
#traffic=df_movement_12.copy()
traffic=df_movement[df_movement.hour_of_day == 18].copy()


In [16]:
# Step 0: Get unique segment definitions
segment_keys = df_movement[["osm_way_id", "osm_start_node_id", "osm_end_node_id"]].drop_duplicates()

In [17]:
segment_keys

,osm_way_id,osm_start_node_id,osm_end_node_id
0,9932085,1236497729,1236497752
3,9932085,81398215,1236497752
9,169753571,1813307792,235637281
33,30709742,235637275,235637281
55,180239867,1906766760,946947901
...,...,...,...
550262,4685726,29784949,29784935
550275,4446505,1277848884,27306422
550299,47481120,604320404,604320405
550302,520751012,29881567,281375696


In [22]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely.ops import linemerge
from collections import defaultdict, deque
from tqdm import tqdm

# === STEP 0: Build OSM mini-segment graph ===
way_segments = defaultdict(lambda: defaultdict(list))

for _, row in segments.iterrows():
    way_id = int(row["osm_way_id"])
    start = int(row["osm_start_node_id"])
    end = int(row["osm_end_node_id"])
    way_segments[way_id][start].append((end, row["geometry"]))

# === STEP 1: Pathfinding function ===
def find_path(way_id, start, end, max_depth=10):
    visited = set()
    queue = deque([(start, [start], [])])
    
    while queue:
        current, path, geoms = queue.popleft()
        if current == end:
            return path, linemerge(geoms)
        if len(path) > max_depth:
            continue
        for neighbor, geom in way_segments[way_id].get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor], geoms + [geom]))
    return None, None

# === STEP 2: Reconstruct geometry per unique segment ===
segment_keys = df_movement[["osm_way_id", "osm_start_node_id", "osm_end_node_id"]].drop_duplicates()
reconstructed_rows = []

for _, row in tqdm(segment_keys.iterrows(), total=len(segment_keys)):
    try:
        way_id = int(row["osm_way_id"])
        start = int(float(row["osm_start_node_id"]))
        end = int(float(row["osm_end_node_id"]))
    except Exception:
        continue

    path = geom = None
    direction = None

    for depth in [6, 12, 18, 24, 50, 100]:
        path, geom = find_path(way_id, start, end, max_depth=depth)
        if geom:
            direction = "forward"
            break

    if not geom:
        for depth in [6, 12, 18, 24, 50, 100]:
            path, geom = find_path(way_id, end, start, max_depth=depth)
            if geom:
                geom = LineString(list(geom.coords)[::-1])
                path = path[::-1]
                direction = "reverse"
                break

    if geom:
        reconstructed_rows.append({
            "osm_way_id": way_id,
            "osm_start_node_id": start,
            "osm_end_node_id": end,
            "geometry": geom,
            "reconstructed_path": path,
            "reconstruction_depth": depth,
            "reconstruction_direction": direction,
        })

# === STEP 3: Merge geometries into full df_movement (all 24 hours) ===
geometry_df = gpd.GeoDataFrame(reconstructed_rows, geometry="geometry", crs=segments.crs)

merged_all = df_movement.merge(
    geometry_df,
    on=["osm_way_id", "osm_start_node_id", "osm_end_node_id"],
    how="left"
)


100%|██████████| 34697/34697 [00:02<00:00, 12384.31it/s]


In [23]:
geometry_df

,osm_way_id,osm_start_node_id,osm_end_node_id,geometry,reconstructed_path,reconstruction_depth,reconstruction_direction
0,9932085,1236497729,1236497752,"LINESTRING (13.19421 52.52895, 13.19461 52.529...","[1236497729, 3990000664, 1236497752]",6,reverse
1,9932085,81398215,1236497752,"LINESTRING (13.1948 52.52906, 13.19471 52.52904)","[81398215, 1236497752]",6,forward
2,169753571,1813307792,235637281,"LINESTRING (13.37817 52.4677, 13.3782 52.46754...","[1813307792, 650378267, 621761883, 235637281]",6,forward
3,30709742,235637275,235637281,"LINESTRING (13.37833 52.46709, 13.37824 52.46729)","[235637275, 235637281]",6,forward
4,192497995,449099355,5252316833,"LINESTRING (13.55065 52.50929, 13.55103 52.50926)","[449099355, 5252316833]",6,forward
...,...,...,...,...,...,...,...
33475,4520013,26848672,26848657,"LINESTRING (13.28717 52.45807, 13.28721 52.45835)","[26848672, 26848657]",6,forward
33476,4615001,29266133,29224784,"LINESTRING (13.41995 52.53173, 13.42016 52.531...","[29266133, 2888909160, 2357289973, 2283081504,...",6,reverse
33477,4446505,1277848884,27306422,"LINESTRING (13.29692 52.51104, 13.29658 52.511...","[1277848884, 4982740894, 3951566061, 27306422]",6,forward
33478,47481120,604320404,604320405,"LINESTRING (13.41929 52.37451, 13.41889 52.37506)","[604320404, 604320405]",6,forward


In [24]:
merged_all

,year,quarter,hour_of_day,segment_id,start_junction_id,end_junction_id,osm_way_id,osm_start_node_id,osm_end_node_id,speed_kph_mean,speed_kph_stddev,speed_kph_p50,speed_kph_p85,geometry,reconstructed_path,reconstruction_depth,reconstruction_direction
0,2019,2,0,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,41.258,9.239,41.330,46.680,"LINESTRING (13.19421 52.52895, 13.19461 52.529...","[1236497729, 3990000664, 1236497752]",6.0,reverse
1,2019,2,23,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,35.794,6.706,35.832,41.398,"LINESTRING (13.19421 52.52895, 13.19461 52.529...","[1236497729, 3990000664, 1236497752]",6.0,reverse
2,2019,2,22,d0034ae2336f81ef5933a211f2ce2d979f0aff2a,66081f9fe2860af2a498e1248334cb51d6ea5073,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,1236497729,1236497752,32.766,13.401,35.171,43.469,"LINESTRING (13.19421 52.52895, 13.19461 52.529...","[1236497729, 3990000664, 1236497752]",6.0,reverse
3,2019,2,23,277640ca389fc7f0fc8a583386e6063df80485f0,ad26106a25d52c3409bd0165f05de3047a4e96b5,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,81398215,1236497752,34.507,4.639,35.207,38.160,"LINESTRING (13.1948 52.52906, 13.19471 52.52904)","[81398215, 1236497752]",6.0,forward
4,2019,2,1,277640ca389fc7f0fc8a583386e6063df80485f0,ad26106a25d52c3409bd0165f05de3047a4e96b5,9c00a9375aeaba9c08f4f2fd507cc1808ba83b89,9932085,81398215,1236497752,41.512,6.049,41.128,47.681,"LINESTRING (13.1948 52.52906, 13.19471 52.52904)","[81398215, 1236497752]",6.0,forward
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550328,2019,2,10,46ec6887faf8d9d9d26fb72f575ae064f38d65dd,3c358a996e6db9c97f327284d8c6cd7355ddc03c,8e9f29c18cf04f6bac662cd03017619dfd68e0ee,385627960,312370453,29858455,41.501,20.236,40.446,61.008,"LINESTRING (13.55115 52.55392, 13.55141 52.554...","[312370453, 3889644211, 29858455]",6.0,forward
550329,2019,2,18,46ec6887faf8d9d9d26fb72f575ae064f38d65dd,3c358a996e6db9c97f327284d8c6cd7355ddc03c,8e9f29c18cf04f6bac662cd03017619dfd68e0ee,385627960,312370453,29858455,47.609,14.407,51.682,58.009,"LINESTRING (13.55115 52.55392, 13.55141 52.554...","[312370453, 3889644211, 29858455]",6.0,forward
550330,2019,2,22,46ec6887faf8d9d9d26fb72f575ae064f38d65dd,3c358a996e6db9c97f327284d8c6cd7355ddc03c,8e9f29c18cf04f6bac662cd03017619dfd68e0ee,385627960,312370453,29858455,48.087,17.279,51.740,62.680,"LINESTRING (13.55115 52.55392, 13.55141 52.554...","[312370453, 3889644211, 29858455]",6.0,forward
550331,2019,2,1,46ec6887faf8d9d9d26fb72f575ae064f38d65dd,3c358a996e6db9c97f327284d8c6cd7355ddc03c,8e9f29c18cf04f6bac662cd03017619dfd68e0ee,385627960,312370453,29858455,48.253,17.988,53.074,64.829,"LINESTRING (13.55115 52.55392, 13.55141 52.554...","[312370453, 3889644211, 29858455]",6.0,forward


In [25]:
merged_all[merged_all.hour_of_day == 18]

,year,quarter,hour_of_day,segment_id,start_junction_id,end_junction_id,osm_way_id,osm_start_node_id,osm_end_node_id,speed_kph_mean,speed_kph_stddev,speed_kph_p50,speed_kph_p85,geometry,reconstructed_path,reconstruction_depth,reconstruction_direction
24,2019,2,18,1f4bccb7a80046e3cd483789f351efe99e612d1a,f61823a7db684f4f2c94e9ff25da7e6710d95c76,32c9d31816ecf0fde2586bab37995d06ad5f6f3e,169753571,1813307792,235637281,33.863,15.467,38.073,48.314,"LINESTRING (13.37817 52.4677, 13.3782 52.46754...","[1813307792, 650378267, 621761883, 235637281]",6.0,forward
46,2019,2,18,f50b8a5ca0afcdd31eb48403172284dbc42f2aeb,dc072f474bcbf30d75bd3b1e42974396af504bbb,32c9d31816ecf0fde2586bab37995d06ad5f6f3e,30709742,235637275,235637281,41.181,9.464,42.334,50.418,"LINESTRING (13.37833 52.46709, 13.37824 52.46729)","[235637275, 235637281]",6.0,forward
68,2019,2,18,afa6bf679eba3d1479ae4c6c8145fcca949abb41,ebb32c0705b8669fa3a650623cd9e5144efc9207,59050a5fd5b647ebf027e5be0ac7af915fd998ac,180239867,1906766760,946947901,92.340,13.108,93.552,103.062,None,NaN,NaN,NaN
75,2019,2,18,fff4ebf924a4525640b0e4a0cf30cd376e3b5732,5055974adf81851fdaa9c67547d7fb6998bdddb9,d57667861dcb1c9d2cdc9d684e408f94c7276d35,192497995,449099355,5252316833,47.938,11.579,50.703,58.167,"LINESTRING (13.55065 52.50929, 13.55103 52.50926)","[449099355, 5252316833]",6.0,forward
98,2019,2,18,2a4cd117e8b758cde331b8fca96ebfa5482a53eb,871dbeee4e706cae8b95be11a608707166267e0c,2d5f878e230c20918c1d335c180e2b041e7c0766,30253522,29826350,4998526579,46.576,7.791,46.704,53.143,"LINESTRING (13.41361 52.48832, 13.41344 52.488...","[29826350, 3828053052, 3888654213, 4998526579]",6.0,forward
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
550248,2019,2,18,2a11749e5df81e3f6a777fbda94ba2bf4a051163,cfd65c8f58632871374653f1066885bc9fb90276,8304333a765fefa4dd0078ea9a021d0fd9beee8d,4615001,29266133,29224784,25.570,5.441,25.144,29.723,"LINESTRING (13.41995 52.53173, 13.42016 52.531...","[29266133, 2888909160, 2357289973, 2283081504,...",6.0,reverse
550256,2019,2,18,8d14db2000da0c540623437ae91f0090d111296e,214aba0ae32c3e7a169ff94eb530d35d3ac3e81c,aa52b3389e94e35b97612908398ae17dbf59cffb,4685726,29784942,29784935,24.401,5.960,23.771,29.908,None,NaN,NaN,NaN
550262,2019,2,18,de0b5bc475e7ec8fa6f1dd4e2bffe3bf54407a14,27c7d760fddbabb9b7951ade69bc6357023285d7,aa52b3389e94e35b97612908398ae17dbf59cffb,4685726,29784949,29784935,26.052,6.460,27.212,31.003,None,NaN,NaN,NaN
550298,2019,2,18,ff8a397700e1b9293bad2c93cd1a03aedf5ae6f5,9a51ec7772c9bbf6375eeace41d2a4a62cca64e3,7205366afb085f05d29e3698481f57b08693cd30,4446505,1277848884,27306422,44.992,8.962,45.242,52.335,"LINESTRING (13.29692 52.51104, 13.29658 52.511...","[1277848884, 4982740894, 3951566061, 27306422]",6.0,forward


In [37]:
# Step 1: Pivot speed by hour
df_pivoted = (
    merged_all
    .pivot_table(
        index=["osm_way_id", "osm_start_node_id", "osm_end_node_id","reconstruction_direction"],
        columns="hour_of_day",
        values="speed_kph_mean"
    )
    .add_prefix("speed_")
    .reset_index()
)

# Step 2: Static geometry + attributes (only one per segment)
geometry_and_static = (
    merged_all
    .drop_duplicates(subset=["osm_way_id", "osm_start_node_id", "osm_end_node_id"])
    .set_index(["osm_way_id", "osm_start_node_id", "osm_end_node_id"])[["geometry"]]
)

# Step 3: Combine into final GeoDataFrame
df_final = gpd.GeoDataFrame(
    df_pivoted.set_index(["osm_way_id", "osm_start_node_id", "osm_end_node_id"]).join(geometry_and_static),
    geometry="geometry",
    crs=geometry_df.crs
).reset_index()


In [38]:
df_final

,osm_way_id,osm_start_node_id,osm_end_node_id,reconstruction_direction,speed_0,speed_1,speed_2,speed_3,speed_4,speed_5,...,speed_15,speed_16,speed_17,speed_18,speed_19,speed_20,speed_21,speed_22,speed_23,geometry
0,4045243,307495922,3155500679,forward,46.972,48.012,48.451,49.332,49.371,52.038,...,40.470,42.606,43.146,43.267,43.998,46.180,46.271,45.838,46.564,"LINESTRING (13.45739 52.51538, 13.45775 52.515..."
1,4045243,1822620447,307495922,forward,48.369,49.128,48.748,49.371,49.868,50.939,...,43.381,44.795,45.445,46.227,47.073,47.541,47.394,47.589,48.155,"LINESTRING (13.45421 52.51571, 13.45439 52.515..."
2,4045656,21441709,30432575,reverse,47.792,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (13.37261 52.59296, 13.3727 52.5931..."
3,4045656,21441714,561503519,reverse,45.078,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.748,"LINESTRING (13.37102 52.59076, 13.37135 52.591..."
4,4045656,30432116,21441714,reverse,27.737,27.876,NaN,35.887,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.348,"LINESTRING (13.36987 52.58936, 13.37082 52.590..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33475,699211935,1799364311,6565966749,reverse,30.548,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,27.289,27.348,27.527,NaN,NaN,26.859,28.673,"LINESTRING (13.409 52.53516, 13.40863 52.53479)"
33476,699217505,268224213,1769691978,forward,57.284,56.909,64.483,55.987,56.715,56.842,...,NaN,NaN,NaN,NaN,53.956,52.637,54.731,50.533,54.905,"LINESTRING (13.52338 52.50963, 13.52376 52.509..."
33477,699286157,30691244,482740246,forward,47.741,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47.561,"LINESTRING (13.54398 52.56803, 13.54414 52.567..."
33478,699295574,248079015,152428728,forward,35.993,42.676,35.638,37.272,40.599,37.561,...,39.568,38.350,39.923,39.936,38.120,39.188,37.026,39.150,39.134,"LINESTRING (13.55819 52.45596, 13.55802 52.45594)"


In [36]:
print(df_final.head())

   osm_way_id  osm_start_node_id  osm_end_node_id  speed_0  speed_1  speed_2  \
0     4045243          307495922       3155500679   46.972   48.012   48.451   
1     4045243         1822620447        307495922   48.369   49.128   48.748   
2     4045656           21441709         30432575   47.792      NaN      NaN   
3     4045656           21441714        561503519   45.078      NaN      NaN   
4     4045656           30432116         21441714   27.737   27.876      NaN   

   speed_3  speed_4  speed_5  speed_6  ...  speed_15  speed_16  speed_17  \
0   49.332   49.371   52.038   51.879  ...    40.470    42.606    43.146   
1   49.371   49.868   50.939   51.181  ...    43.381    44.795    45.445   
2      NaN      NaN      NaN      NaN  ...       NaN       NaN       NaN   
3      NaN      NaN      NaN      NaN  ...       NaN       NaN       NaN   
4   35.887      NaN      NaN      NaN  ...       NaN       NaN       NaN   

   speed_18  speed_19  speed_20  speed_21  speed_22  speed_23 

In [43]:
# Melt wide to long
df_long = df_final.melt(
    id_vars=["osm_way_id", "osm_start_node_id", "osm_end_node_id", "reconstruction_direction","geometry"],
    value_vars=[col for col in df_final.columns if col.startswith("speed_")],
    var_name="hour_col",
    value_name="speed_kph_mean"
)

# Extract actual hour from 'speed_0' → 0
df_long["hour_of_day"] = df_long["hour_col"].str.extract(r"speed_(\d+)").astype(int)
df_long = df_long.drop(columns="hour_col")

# Convert to GeoDataFrame again
gdf_long = gpd.GeoDataFrame(df_long, geometry="geometry", crs=df_final.crs)


In [44]:
gdf_long

,osm_way_id,osm_start_node_id,osm_end_node_id,reconstruction_direction,geometry,speed_kph_mean,hour_of_day
0,4045243,307495922,3155500679,forward,"LINESTRING (13.45739 52.51538, 13.45775 52.515...",46.972,0
1,4045243,1822620447,307495922,forward,"LINESTRING (13.45421 52.51571, 13.45439 52.515...",48.369,0
2,4045656,21441709,30432575,reverse,"LINESTRING (13.37261 52.59296, 13.3727 52.5931...",47.792,0
3,4045656,21441714,561503519,reverse,"LINESTRING (13.37102 52.59076, 13.37135 52.591...",45.078,0
4,4045656,30432116,21441714,reverse,"LINESTRING (13.36987 52.58936, 13.37082 52.590...",27.737,0
...,...,...,...,...,...,...,...
803515,699211935,1799364311,6565966749,reverse,"LINESTRING (13.409 52.53516, 13.40863 52.53479)",28.673,23
803516,699217505,268224213,1769691978,forward,"LINESTRING (13.52338 52.50963, 13.52376 52.509...",54.905,23
803517,699286157,30691244,482740246,forward,"LINESTRING (13.54398 52.56803, 13.54414 52.567...",47.561,23
803518,699295574,248079015,152428728,forward,"LINESTRING (13.55819 52.45596, 13.55802 52.45594)",39.134,23


In [53]:
gdf_long = gdf_long[~gdf_long["speed_kph_mean"].isna()]
gdf_long[gdf_long.hour_of_day == 2]

,osm_way_id,osm_start_node_id,osm_end_node_id,reconstruction_direction,geometry,speed_kph_mean,hour_of_day
66960,4045243,307495922,3155500679,forward,"LINESTRING (13.45739 52.51538, 13.45775 52.515...",48.451,2
66961,4045243,1822620447,307495922,forward,"LINESTRING (13.45421 52.51571, 13.45439 52.515...",48.748,2
66967,4054013,28795932,385448884,forward,"LINESTRING (13.51476 52.45303, 13.51528 52.45267)",58.170,2
66968,4054013,292910412,489899272,forward,"LINESTRING (13.51873 52.45051, 13.51983 52.44982)",60.133,2
66969,4054013,385448884,534019714,forward,"LINESTRING (13.51528 52.45267, 13.51563 52.45243)",59.430,2
...,...,...,...,...,...,...,...
100428,698150954,6556641546,6556641548,forward,"LINESTRING (13.41118 52.48852, 13.412 52.48841...",42.981,2
100429,698801383,6561995746,27009012,forward,"LINESTRING (13.30028 52.54782, 13.29993 52.54738)",73.222,2
100436,699217505,268224213,1769691978,forward,"LINESTRING (13.52338 52.50963, 13.52376 52.509...",64.483,2
100438,699295574,248079015,152428728,forward,"LINESTRING (13.55819 52.45596, 13.55802 52.45594)",35.638,2


In [55]:
gdf_long.to_file("df_movement_osm_q2_2019_allHours.fgb", driver="FlatGeobuf")

In [56]:
import subprocess

input_fgb = "df_movement_osm_q2_2019_allHours.fgb"

cmd = [
    "tippecanoe",
    "-o", "uber_movement_osm_q2_2019_allHoures_osm200101.pmtiles",
    "--minimum-zoom=11",
    "--maximum-zoom=15",
    "--drop-rate=0",
    "--drop-densest-as-needed",
    "--no-feature-limit",
    "--no-tile-size-limit",
    "--maximum-tile-bytes=1000000",
    "--force",
    "-l", "uber_movement_osm",
    input_fgb
]

print("Running tippecanoe for health...")
subprocess.run(cmd, check=True)


Running tippecanoe for health...


detected indexed FlatGeobuf: assigning feature IDs by sequence
536759 features, 35399748 bytes of geometry and attributes, 3483933 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
  99.9%  15/17604/10749  


CompletedProcess(args=['tippecanoe', '-o', 'uber_movement_osm_q2_2019_allHoures_osm200101.pmtiles', '--minimum-zoom=11', '--maximum-zoom=15', '--drop-rate=0', '--drop-densest-as-needed', '--no-feature-limit', '--no-tile-size-limit', '--maximum-tile-bytes=1000000', '--force', '-l', 'uber_movement_osm', 'df_movement_osm_q2_2019_allHours.fgb'], returncode=0)